# Recompute manuscript neural metrics

The figure notebooks start from the frozen analysis source tables. The original neural-computation functions used to create those inputs are included below. Recomputing them requires aligned trial-level neural and position data in addition to the source-table dataset. See `data/README.md` for their contracts. All outputs must be saved to a new directory; do not replace released input data.

Both Figure-8 and T-maze manuscript manifold analyses use `legacy_pooled` covariance, 50 neurons, 15-dimensional SpectralEmbedding, 100 repeats and 100 shuffles. These choices are explicit because the historical shared library has other defaults. The 3-dimensional representative visualizations are distinct from this quantitative analysis.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = ROOT / "outputs"
REFIT_STATISTICS = True
from paper import run_figure


Figure-8 manifold calculation. `SUBJECT_DIR` must contain the corresponding `snarm_1d.npy`, `trial_labels.npy` and available context/segment metadata. Choose a mouse from the frozen cohort. Leave this opt-in cell disabled until the aligned inputs are present.

In [ ]:
RUN_NEURAL_RECOMPUTATION = False
if RUN_NEURAL_RECOMPUTATION:
    import numpy as np
    from maze_analysis.manifold import compute_manifold_quantification_for_subject
    SUBJECT_DIR = DATA_ROOT / "aligned_figure8" / "A4"
    result = compute_manifold_quantification_for_subject(
        SUBJECT_DIR, "A4", num_day=5, target_neuron_count=50,
        embedding_dim=15, n_repeat=100, n_shuffle=100,
        seed=2026, reducer="spectral", zscore=True,
        dprime_covariance="legacy_pooled", save_cache=False,
    )
    destination = OUTPUT_ROOT / "recomputed_metrics"
    destination.mkdir(parents=True, exist_ok=True)
    np.save(destination / "A4_manifold.npy", result)


T-maze pooled-covariance manifold calculation. The original implementation retains sample/choice phases, correct-trial rules, route-balanced undersampling and historical subject seeds.

In [ ]:
if RUN_NEURAL_RECOMPUTATION:
    from maze_analysis.tmaze_legacy_pooled import (
        compute_subject_legacy_pooled, cache_to_bin_table,
        bin_to_section_table, historical_subject_seed,
    )
    mouse, group = "P1", "P"
    result = compute_subject_legacy_pooled(
        DATA_ROOT / "aligned_tmaze" / mouse, mouse=mouse, group=group,
        seed=historical_subject_seed(mouse),
    )
    bins = cache_to_bin_table(result)
    bins.to_csv(destination / f"{mouse}_manifold_bins.csv", index=False)
    bin_to_section_table(bins).to_csv(destination / f"{mouse}_manifold_sections.csv", index=False)


The remaining included computation APIs are listed here so their input and output interfaces can be inspected. Figure-8 stability uses condition-ordinal odd/even splitting and fixed-50-neuron PV sampling. T-maze PV separation uses its original route-balanced shuffle implementation. Bayesian decoding uses 98 physical bins, five complete-trial folds and 100 within-trial circular label shifts; the source-data dataset already contains the completed outputs. No CEBRA, cross-validated PCA manifold or KNN result is part of this release.

In [ ]:
from maze_analysis.pv_pc_subjects import (
    compute_pv_within_reliability_subject,
    compute_intraday_pc_within_subject,
    compute_interday_stability_subject,
)
from maze_analysis.pv_shuffle import compute_pv_shuffle_control_for_subject
from t_maze.pv_shuffle import compute_pv_shuffle_control_for_subject as compute_tmaze_pv
from maze_analysis.bayesian_position import prepare_binned_trials, shift_bins_within_trials
from maze_analysis.bayesian_position_batch import TrialGaussianCV, make_trial_folds
from maze_analysis.position_posterior import mean_posterior_matrix, average_mouse_matrices
